# EXACT API navigation

Minimal requests-only notebook for the HS Flensburg EXACT instance. It lists image sets, lists images inside image set `245`, and defines download helpers for thumbnails and original WSI files.


In [ ]:
from pathlib import Path
import os
import re

import pandas as pd
import requests
from requests.auth import HTTPBasicAuth


## Connection settings

`EXACT_PASSWORD` is read from your process environment. It is not stored by `os.getenv`; Python just receives a copy of the value for this running notebook process. Start Jupyter with `EXACT_PASSWORD` set before running this notebook.


In [ ]:
EXACT_BASE_URL = "https://exact.hs-flensburg.de"
EXACT_USERNAME = os.getenv("EXACT_USERNAME", "alha7503")
EXACT_PASSWORD = os.getenv("EXACT_PASSWORD")
if not EXACT_PASSWORD:
    raise RuntimeError("Set EXACT_PASSWORD in your environment before running this notebook.")

exact_session = requests.Session()
exact_session.auth = HTTPBasicAuth(EXACT_USERNAME, EXACT_PASSWORD)


In [ ]:
def exact_url(path):
    if path.startswith("http://") or path.startswith("https://"):
        return path
    return f"{EXACT_BASE_URL.rstrip('/')}/{path.lstrip('/')}"


def exact_get(path, **params):
    response = exact_session.get(exact_url(path), params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def exact_download(path, target_path, **params):
    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)

    with exact_session.get(exact_url(path), params=params, stream=True, timeout=120) as response:
        response.raise_for_status()
        with target_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    return target_path


def exact_get_all(path, limit=100, **params):
    rows = []
    offset = 0

    while True:
        page = exact_get(path, limit=limit, offset=offset, **params)
        rows.extend(page["results"])
        if not page.get("next"):
            break
        offset += limit

    return rows


def safe_filename(value):
    value = str(value or "download")
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._") or "download"


## Smoke test


In [ ]:
api_root = exact_get("/api/v1/")

print(f"Connected to {EXACT_BASE_URL}")
print(f"API resources available: {len(api_root)}")


## Image sets


In [ ]:
image_sets = exact_get_all(
    "/api/v1/images/image_sets/",
    fields="id,name,images",
)

image_sets_df = pd.DataFrame(
    {
        "id": image_set["id"],
        "name": image_set["name"],
        "image_count": len(image_set.get("images") or []),
    }
    for image_set in image_sets
)

image_sets_df.sort_values(["name", "id"]).reset_index(drop=True)


## Images in image set 245


In [ ]:
IMAGE_SET_ID = 245

image_set = exact_get(
    f"/api/v1/images/image_sets/{IMAGE_SET_ID}/",
    fields="id,name,images",
)

images = exact_get_all(
    "/api/v1/images/images/",
    image_set=IMAGE_SET_ID,
    fields="id,name,filename,width,height,mpp,objective_power,image_set",
)

images_df = pd.DataFrame(images)

print(f"Image set {image_set['id']}: {image_set['name']}")
print(f"Images available: {len(images_df)}")

images_df


## Inspect one image


In [ ]:
IMAGE_ID = 9988

image_detail = exact_get(f"/api/v1/images/images/{IMAGE_ID}/")
slide_information = exact_get(f"/api/v1/images/images/{IMAGE_ID}/slide_information/")

image_detail


## Download helpers

The thumbnail download is small and useful as a storage proof. The original WSI download can be very large, so the call is defined but left for you to run deliberately.


In [ ]:
DOWNLOAD_DIR = Path("data/exact")


def image_filename(image):
    name = image.get("filename") or image.get("name") or f"image_{image['id']}"
    return safe_filename(name)


def download_thumbnail_by_id(image_id, download_dir=DOWNLOAD_DIR / "thumbnails"):
    image = exact_get(f"/api/v1/images/images/{image_id}/", fields="id,name,filename")
    target_path = Path(download_dir) / f"{image['id']}_{Path(image_filename(image)).stem}_thumbnail.png"
    return exact_download(f"/api/v1/images/images/{image_id}/thumbnail/", target_path)


def download_image_by_id(image_id, download_dir=DOWNLOAD_DIR / "images", original_image=True):
    image = exact_get(f"/api/v1/images/images/{image_id}/", fields="id,name,filename")
    target_path = Path(download_dir) / f"{image['id']}_{image_filename(image)}"
    return exact_download(
        f"/images/api/image/download/{image_id}/",
        target_path,
        original_image=original_image,
    )


In [ ]:
thumbnail_path = download_thumbnail_by_id(IMAGE_ID)
thumbnail_path


## Download original WSI

Run this only when you actually want the huge scan on disk.


In [ ]:
# original_image_path = download_image_by_id(IMAGE_ID)
# original_image_path
